# Fermion-to-Qubit Mappings

Build the Jordan-Wigner transformation from scratch in numpy, verify it preserves fermionic structure, then contrast it qualitatively with Bravyi-Kitaev and parity mapping.

**Objectives:**
- Construct the JW image of creation and annihilation operators on 4 modes as dense 16x16 matrices, including the all-important Z-string
- Verify the canonical anticommutation relations `{a_p, a_q^dagger} = delta_pq I` and `{a_p, a_q} = 0` directly from the matrices
- Confirm the JW number operator `N_p = a_p^dagger a_p` is diagonal with 0/1 entries and reads occupation correctly
- Map a small fermionic hopping/number Hamiltonian to Pauli strings and inspect their weight
- Understand why Bravyi-Kitaev (O(log n) weight) and parity mapping (qubit tapering) exist

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

device = LocalSimulator()

## 1. The problem: qubits do not speak fermion

Second quantization tracks *occupation* — for each spin-orbital `p`, an annihilation operator `a_p` removes an electron and a creation operator `a_p^dagger` adds one. The physics that makes electrons electrons is the **canonical anticommutation relations** (CAR):

$$
\{a_p,\, a_q^\dagger\} = \delta_{pq}\, I, \qquad \{a_p,\, a_q\} = 0 .
$$

Here `{A, B} = AB + BA`. The `{a_p, a_p} = 0` case is the Pauli exclusion principle written as algebra: you cannot create the same electron twice.

Qubits have no such rule. Flipping qubit 3 with a bare `X` ignores qubits 0 through 2 entirely — it carries no minus signs. A **fermion-to-qubit mapping** is a recipe that reproduces the anticommutation relations using qubit operators, smuggling the missing signs back in.

We work with `N_MODES = 4` modes, so every operator is a `2**4 = 16`-dimensional dense matrix. We follow the qcsim convention throughout: **big-endian, qubit 0 is the leftmost (most-significant) tensor factor**, so the basis ket `|1100>` means modes 0 and 1 occupied, modes 2 and 3 empty.

In [ ]:
# Single-qubit Pauli matrices and identity (complex, so i*Y stays exact).
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

N_MODES = 4
DIM = 2 ** N_MODES  # 16


def kron_list(ops):
    """Tensor (Kronecker) product of a list of single-qubit operators.

    Big-endian: ops[0] is the leftmost / most-significant factor (qubit 0).
    """
    out = np.array([[1.0]], dtype=complex)
    for op in ops:
        out = np.kron(out, op)
    return out


def pauli_on(P, target, n=N_MODES):
    """Single Pauli P acting on `target`, identity elsewhere."""
    return kron_list([P if q == target else I2 for q in range(n)])


# Sanity: Z on qubit 0 flips the sign of exactly the half of the basis where
# qubit 0 is |1> (the high indices, since qubit 0 is the most-significant bit).
print("pauli_on(Z, 0) diagonal:", np.real(np.diag(pauli_on(Z, 0))).astype(int))
print("pauli_on(Z, 3) diagonal:", np.real(np.diag(pauli_on(Z, 3))).astype(int))

## 2. Building the Jordan-Wigner operators

The Jordan-Wigner transformation assigns occupation of mode `p` to the state of qubit `p`: `|0>` empty, `|1>` occupied. The local part of an annihilation operator is the lowering operator

$$
\sigma^- = \frac{X + iY}{2} = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix},
$$

which sends `|1> -> |0>`. (Its conjugate transpose, the raising operator `(X - iY)/2`, is the creation piece.) But a bare local lowering operator does **not** anticommute across modes. The fix is the **Z-string**: every annihilation operator on mode `p` carries a chain of `Z` operators on all *lower-index* qubits `0, 1, ..., p-1`:

$$
a_p = \Big(\underbrace{Z \otimes Z \otimes \cdots \otimes Z}_{q < p}\Big) \otimes \sigma^-_p \otimes \underbrace{I \otimes \cdots \otimes I}_{q > p}.
$$

That chain of `Z` operators *is* fermionic antisymmetry made concrete: it reads the parity of every mode below `p` and stamps the right sign. The cost is locality — an operator on mode `p` now touches every qubit beneath it.

In [ ]:
def jw_annihilation(p, n=N_MODES):
    """Jordan-Wigner image of the annihilation operator a_p as a dense matrix.

    a_p = Z_0 Z_1 ... Z_{p-1}  (X_p + i Y_p)/2.
    The Z-string runs over all LOWER-index qubits (q < p).
    """
    lowering = (X + 1j * Y) / 2  # sigma^-, sends |1> -> |0>
    factors = []
    for q in range(n):
        if q < p:
            factors.append(Z)        # Z-string on lower-index qubits
        elif q == p:
            factors.append(lowering)
        else:
            factors.append(I2)       # identity above
    return kron_list(factors)


def jw_creation(p, n=N_MODES):
    """Creation operator a_p^dagger is the conjugate transpose of a_p."""
    return jw_annihilation(p, n).conj().T


# Each a_p^dagger creates in mode p: apply to vacuum |0000> and read the ket.
for p in range(N_MODES):
    vac = np.zeros(DIM, dtype=complex)
    vac[0] = 1.0
    created = jw_creation(p) @ vac
    idx = int(np.argmax(np.abs(created)))
    ket = format(idx, "04b")
    sign = np.sign(np.real(created[idx]))
    print(f"a_{p}^dagger |0000> = (sign {sign:+.0f}) |{ket}>")

Each `a_p^dagger` drops an electron into mode `p`, but the Z-string means the resulting amplitude can pick up a sign depending on what is already occupied below `p`. That sign bookkeeping is exactly what makes the operators anticommute. Let us check that directly.

## 3. Verifying the anticommutation relations

The whole point of the mapping is that it reproduces the CAR. We check, for every `(p, q)` pair, that

- `{a_p, a_q^dagger} = delta_pq * I`  (identity when `p == q`, zero otherwise), and
- `{a_p, a_q} = 0`  (always zero, including `p == q`).

These are exact matrix identities — we assert closeness to the expected `16x16` matrices, never on sampling.

In [ ]:
def anticommutator(A, B):
    return A @ B + B @ A


IDENT = np.eye(DIM, dtype=complex)
ZERO = np.zeros((DIM, DIM), dtype=complex)

# Check {a_p, a_q^dagger} = delta_pq I over all pairs.
for p in range(N_MODES):
    for q in range(N_MODES):
        ac = anticommutator(jw_annihilation(p), jw_creation(q))
        expected = IDENT if p == q else ZERO
        assert np.allclose(ac, expected), f"{{a_{p}, a_{q}^dagger}} wrong"

# Check {a_p, a_q} = 0 over all pairs (includes the exclusion case p == q).
for p in range(N_MODES):
    for q in range(N_MODES):
        ac = anticommutator(jw_annihilation(p), jw_annihilation(q))
        assert np.allclose(ac, ZERO), f"{{a_{p}, a_{q}}} not zero"

# Spot-print a couple of representative pairs.
diag_pair = anticommutator(jw_annihilation(1), jw_creation(1))
off_pair = anticommutator(jw_annihilation(0), jw_creation(2))
print("{a_1, a_1^dagger} == I :", np.allclose(diag_pair, IDENT))
print("{a_0, a_2^dagger} == 0 :", np.allclose(off_pair, ZERO))
print("{a_0, a_0}        == 0 :", np.allclose(anticommutator(jw_annihilation(0), jw_annihilation(0)), ZERO))
print("\nAll canonical anticommutation relations verified for 4 modes.")

The Z-string is doing real work here. Without it, `a_0` and `a_1` would *commute* (two local lowering operators on different qubits always do), and the algebra would be bosonic, not fermionic. With it, the `Z` on qubit 0 inside `a_1` anticommutes with the `sigma^-` on qubit 0 inside `a_0`, flipping a `+` into a `-` so the anticommutator collapses to zero.

## 4. The number operator reads occupation

The occupation of mode `p` is measured by the number operator `N_p = a_p^dagger a_p`. For a valid mapping it must be:

- **diagonal** in the computational basis (occupation is a classical readout), and
- have only **0/1 entries** (a spin-orbital holds zero or one electron).

We assert both, then confirm `N_p` reads the right answer on an occupied reference state.

In [ ]:
number_ops = [jw_creation(p) @ jw_annihilation(p) for p in range(N_MODES)]

for p, Np in enumerate(number_ops):
    # Diagonal: off-diagonal part must vanish.
    assert np.allclose(Np, np.diag(np.diag(Np))), f"N_{p} not diagonal"
    diag = np.real(np.diag(Np))
    # Entries are exactly 0 or 1.
    assert np.allclose(diag, np.round(diag)), f"N_{p} not integer-valued"
    assert set(np.round(diag).astype(int).tolist()) <= {0, 1}, f"N_{p} not 0/1"

print("Every N_p is diagonal with 0/1 entries.\n")

# Build the occupied reference ket |1100>: modes 0,1 occupied (big-endian, qubit 0 leftmost).
occupied_idx = int("1100", 2)  # = 12
psi = np.zeros(DIM, dtype=complex)
psi[occupied_idx] = 1.0

# N_p |occupied> = (occupation of p) |occupied>: read each occupation number.
occupations = [int(round(np.real(psi.conj() @ (Np @ psi)))) for Np in number_ops]
print("Reference state |1100> occupation per mode:", occupations)
assert occupations == [1, 1, 0, 0]

# N_0 acts as the identity on this state (mode 0 is occupied): N_0|psi> = |psi>.
assert np.allclose(number_ops[0] @ psi, psi)
print("N_0 |1100> = |1100>  (eigenvalue 1, mode 0 occupied):", np.allclose(number_ops[0] @ psi, psi))

total_number = sum(number_ops)
print("Total electron number on |1100> =", int(round(np.real(psi.conj() @ (total_number @ psi)))))

## 5. Mapping a fermionic Hamiltonian to Pauli strings

Now the payoff: take a small second-quantized Hamiltonian and push it through Jordan-Wigner to get a sum of Pauli strings — the form a quantum device can actually measure. We use a two-mode **hopping + on-site** model embedded in our 4-mode register:

$$
H = t\,(a_0^\dagger a_1 + a_1^\dagger a_0) \;+\; \mu\,(a_0^\dagger a_0 + a_1^\dagger a_1).
$$

The hopping term moves an electron between modes 0 and 1; the on-site term counts electrons. We build the dense matrix from our JW operators, then decompose it into Pauli strings by projecting onto each `P` with the Hilbert-Schmidt inner product `coeff = Tr(P^dagger H) / DIM`.

In [ ]:
t, mu = 0.5, 1.0

a0, a1 = jw_annihilation(0), jw_annihilation(1)
a0d, a1d = jw_creation(0), jw_creation(1)

H = t * (a0d @ a1 + a1d @ a0) + mu * (a0d @ a0 + a1d @ a1)
assert np.allclose(H, H.conj().T), "Hamiltonian must be Hermitian"

PAULI = {"I": I2, "X": X, "Y": Y, "Z": Z}


def pauli_string(labels):
    return kron_list([PAULI[ch] for ch in labels])


def pauli_decompose(M, n=N_MODES, tol=1e-9):
    """Decompose a 2^n x 2^n Hermitian matrix into Pauli strings."""
    from itertools import product
    terms = []
    for labels in product("IXYZ", repeat=n):
        P = pauli_string(labels)
        coeff = np.trace(P.conj().T @ M) / (2 ** n)
        if abs(coeff) > tol:
            terms.append(("".join(labels), coeff))
    return terms


def pauli_weight(label):
    """Number of non-identity factors (locality / weight) of a Pauli string."""
    return sum(1 for ch in label if ch != "I")


terms = pauli_decompose(H)

print(f"H maps to {len(terms)} Pauli terms:\n")
for label, coeff in terms:
    print(f"  {label}   coeff = {coeff.real:+.4f}   weight = {pauli_weight(label)}")

# Reconstruct and verify the decomposition is exact.
H_rebuilt = sum(c * pauli_string(l) for l, c in terms)
assert np.allclose(H_rebuilt, H), "Pauli decomposition does not reconstruct H"
print("\nDecomposition reconstructs H exactly:", np.allclose(H_rebuilt, H))

Two things to read off this output:

- The **on-site number terms** become single-`Z` strings (`ZIII`, `IZII`): `N_p = (I - Z_p)/2`, weight 1.
- The **hopping term** `a_0^dagger a_1 + h.c.` becomes `(XX + YY)/2` on the *adjacent* modes 0 and 1 (`XXII`, `YYII`), weight 2. The Z-strings of `a_0^dagger` and `a_1` overlap on no intermediate qubit here, so they cancel and the result stays weight-2.

That cancellation is special to *adjacent* modes. A hopping term between distant modes `p` and `r` drags a Z-string across every qubit in between — and that is the locality cost that motivates the next section.

## 6. Why the Z-string hurts: locality grows with distance

Jordan-Wigner is exact and intuitive, but its Pauli weight is its weakness. The annihilation operator `a_p` already has a Z-string of length `p`. A hopping term `a_p^dagger a_r` between non-adjacent modes leaves a residual Z-string across the modes strictly between `p` and `r`, so its Pauli weight scales **linearly** with `|p - r|`. On hardware, every extra Pauli factor is another qubit to measure and another opportunity for error.

We can see the growth directly: decompose `a_p^dagger a_r + a_r^dagger a_p` for widening gaps and read off the maximum weight.

In [ ]:
print("Hopping term  a_p^dagger a_r + h.c.  ->  max Pauli weight (Jordan-Wigner):\n")
for (p, r) in [(0, 1), (0, 2), (0, 3)]:
    hop = jw_creation(p) @ jw_annihilation(r) + jw_creation(r) @ jw_annihilation(p)
    hop_terms = pauli_decompose(hop)
    max_w = max(pauli_weight(lbl) for lbl, _ in hop_terms)
    labels = ", ".join(lbl for lbl, _ in hop_terms)
    print(f"  modes ({p},{r}) gap={r - p}:  max weight = {max_w}   terms: {labels}")

print(
    "\nWeight grows with the gap: that linear Z-string is exactly what "
    "Bravyi-Kitaev trades away."
)

## 7. The alternatives: Bravyi-Kitaev and parity mapping (qualitative)

Jordan-Wigner stores **occupation** locally and pays for **parity** with a linear Z-string. Two other mappings rebalance that ledger:

- **Bravyi-Kitaev.** Stores a clever mixture of occupation and partial parity in each qubit, arranged on a binary tree. Both reading occupation and updating parity then touch only `O(log n)` qubits instead of `O(n)`. Concretely, a creation/annihilation operator maps to Pauli strings of weight `O(log n)` rather than `O(n)` — a real win as molecules grow, at the price of an index scheme that is far less transparent than "qubit p = mode p."

- **Parity mapping.** Stores the *cumulative* parity up to each mode directly. This makes two physical symmetries — total electron number and total spin — explicit in two specific qubits. Because those qubits then carry no entangling information, they can be **tapered off** entirely. This is precisely the move that collapses the 4-qubit, 15-term H2 Hamiltonian in the GUIDE down to a **single qubit** with 3 terms.

We illustrate the headline locality contrast with the standard scaling laws (the actual BK construction is beyond a from-scratch numpy cell, but the asymptotics are the point):

In [ ]:
# Illustrative locality comparison: JW (linear) vs BK (logarithmic) operator weight.
# JW weight for a_p is the Z-string length p+1; BK is ~log2 scaling.
modes = np.arange(1, 33)
jw_weight = modes                                  # ~ n   (Z-string)
bk_weight = np.ceil(np.log2(modes + 1)) + 1        # ~ log n

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(modes, jw_weight, "o-", label="Jordan-Wigner  ~ O(n)", color="#c0392b")
ax.plot(modes, bk_weight, "s-", label="Bravyi-Kitaev  ~ O(log n)", color="#2471a3")
ax.set_xlabel("mode index p (system size)")
ax.set_ylabel("Pauli weight of a_p (qubits touched)")
ax.set_title("Operator locality: linear Z-string vs logarithmic tree")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At n = 32 modes:  JW weight ~", int(jw_weight[-1]),
      " vs  BK weight ~", int(bk_weight[-1]))

## 8. Connecting the mapping to a circuit

A Jordan-Wigner operator is not just a matrix — it tells you which qubits a gate must touch. The Hartree-Fock reference state `|1100>` (modes 0 and 1 occupied) is prepared by flipping qubits 0 and 1 from `|0000>`, which is exactly `a_0^dagger a_1^dagger` acting on the vacuum. We build that state as a Braket circuit and confirm its state vector matches the ket our number operators read in Section 4.

In [ ]:
# Prepare the Hartree-Fock reference |1100> on a 4-qubit register.
hf = Circuit()
hf.x(0)  # occupy mode 0
hf.x(1)  # occupy mode 1
# Touch qubits 2 and 3 with identity so the state vector spans all 4 qubits.
hf.i(2)
hf.i(3)

sv = np.array(hf.state_vector())
peak = int(np.argmax(np.abs(sv)))
print("Hartree-Fock circuit prepares |%s>" % format(peak, "04b"))

# It must be exactly the |1100> basis vector our N_p operators measured earlier.
expected = np.zeros(DIM, dtype=complex)
expected[int("1100", 2)] = 1.0
assert np.allclose(sv, expected), "circuit state does not match |1100>"

# Energy of this reference under H from Section 5: <psi|H|psi>, computed in numpy.
hf_energy = np.real(sv.conj() @ (H @ sv))
print("State vector matches the |1100> ket from Section 4:", np.allclose(sv, expected))
print("Reference energy <1100|H|1100> = %.4f" % hf_energy)
# Both modes occupied -> on-site mu each, no hopping contribution on a number eigenstate.
assert np.isclose(hf_energy, 2 * mu)

## Exercises

Try these by extending the cells above. Keep everything in pure numpy and assert against exact matrices or state vectors.

In [ ]:
# Exercise 1: Verify {a_p^dagger, a_q^dagger} = 0 for all p, q (the creation-side
# exclusion relation). Mirror the loop in Section 3 using jw_creation.
#
# for p in range(N_MODES):
#     for q in range(N_MODES):
#         ...  # assert anticommutator(jw_creation(p), jw_creation(q)) is ZERO


# Exercise 2: Add a two-body interaction term  V * N_0 * N_1  to the Hamiltonian H
# in Section 5, re-decompose into Pauli strings, and confirm a weight-2 ZZ term
# appears. (Hint: N_p = number_ops[p].)
#
# V = 0.8
# H2body = H + V * (number_ops[0] @ number_ops[1])
# ...  # pauli_decompose(H2body) and look for "ZZII"


# Exercise 3: Build the dense matrix of a single distant hopping operator
# a_0^dagger a_3 and print its Pauli decomposition. Identify the residual
# Z-string on the intermediate qubits 1 and 2.
#
# distant = jw_creation(0) @ jw_annihilation(3)
# ...  # pauli_decompose(distant + distant.conj().T)


# Exercise 4: Diagonalize H (np.linalg.eigvalsh) and compare its ground-state
# energy to the reference energy <1100|H|1100> from Section 8. Which is lower,
# and why? (Hint: hopping mixes occupied/virtual configurations.)

## Summary

- The **Jordan-Wigner transformation** maps mode occupation to qubit state and reproduces fermionic anticommutation by attaching a **Z-string** of `Z` operators on all lower-index qubits to each creation/annihilation operator.
- We built `a_p` and `a_p^dagger` as dense `16x16` matrices on 4 modes and verified the **canonical anticommutation relations** `{a_p, a_q^dagger} = delta_pq I` and `{a_p, a_q} = 0` exactly.
- The **number operator** `N_p = a_p^dagger a_p` is diagonal with 0/1 entries and reads occupation correctly: `N_p|occupied> = |occupied>`.
- Mapping a hopping/number Hamiltonian gives **Pauli strings** — number terms become single-`Z`, adjacent hopping becomes `(XX + YY)/2` — and the Z-string makes weight grow **linearly** with hopping distance.
- **Bravyi-Kitaev** trades that linear weight for `O(log n)`, and **parity mapping** exposes symmetries that let you **taper** qubits away — the move that shrinks H2 from four qubits to one.

**Next:** [`03-vqe-h2.ipynb`](03-vqe-h2.ipynb) — put a mapped Hamiltonian to work, minimizing `<psi(theta)|H|psi(theta)>` to find the H2 ground-state energy.